In [1]:
# sol1.ipynb - Full pipeline with GIN + fingerprint/descriptor embeddings + cosine similarity features
import numpy as np
import pandas as pd
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GINConv, global_add_pool, global_mean_pool

from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem import rdMolDescriptors

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler, normalize
from tqdm import tqdm
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


/Users/wiktoriagrodzka/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Using device: cpu


In [5]:
# =====================
# 1) Load data
# =====================
DATA_DIR = ""

# Adjust columns if needed based on actual dataset naming
train_path = f"chebi_dataset_train.parquet"
test_path = f"chebi_dataset_test_empty.parquet"

# NOTE: columns are assumed to be: id, smiles, class_* or mol_id, SMILES
# Adjust here if needed

df_train = pd.read_parquet(train_path)
df_test = pd.read_parquet(test_path)

# Normalize column names
if "SMILES" in df_train.columns:
    smiles_col = "SMILES"
elif "smiles" in df_train.columns:
    smiles_col = "smiles"
else:
    raise ValueError("SMILES column not found")

if "mol_id" in df_train.columns:
    id_col = "mol_id"
elif "id" in df_train.columns:
    id_col = "id"
else:
    raise ValueError("ID column not found")

label_cols = [c for c in df_train.columns if c.startswith("class_")]
num_classes = len(label_cols)
print(f"Train: {len(df_train)} molecules, {num_classes} classes")
print(f"Test:  {len(df_test)} molecules")


Train: 33668 molecules, 500 classes
Test:  11223 molecules


In [6]:
# =====================
# 2) Parse ontology (DAG)
# =====================

def parse_obo(path):
    child_to_parents = defaultdict(list)
    current_id = None
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line.startswith("id: class_"):
                current_id = line.split("id: ")[1]
            elif line.startswith("is_a: class_"):
                parent = line.split("is_a: ")[1].strip().split()[0]
                child_to_parents[current_id].append(parent)
    return child_to_parents

child_to_parents = parse_obo(f"chebi_classes.obo")

# Parent mask
class_to_idx = {c: i for i, c in enumerate(label_cols)}
parent_mask = torch.zeros(num_classes, num_classes)
for child, parents in child_to_parents.items():
    if child in class_to_idx:
        for p in parents:
            if p in class_to_idx:
                parent_mask[class_to_idx[child], class_to_idx[p]] = 1.0
parent_mask_np = parent_mask.numpy()
print(f"Hierarchy: {int(parent_mask.sum())} direct parent-child edges")

# For fast ancestor closure
parent_list = {class_to_idx[c]: [class_to_idx[p] for p in ps if p in class_to_idx]
               for c, ps in child_to_parents.items() if c in class_to_idx}


Hierarchy: 748 direct parent-child edges


In [7]:
# =====================
# 3) SMILES -> PyG graph
# =====================
ATOM_FEATURES = {
    'atomic_num': list(range(1, 119)),
    'degree': [0, 1, 2, 3, 4, 5],
    'formal_charge': [-2, -1, 0, 1, 2],
    'num_hs': [0, 1, 2, 3, 4],
    'hybridization': [
        Chem.rdchem.HybridizationType.SP,
        Chem.rdchem.HybridizationType.SP2,
        Chem.rdchem.HybridizationType.SP3,
        Chem.rdchem.HybridizationType.SP3D,
        Chem.rdchem.HybridizationType.SP3D2,
    ],
}

BOND_TYPES = [
    Chem.rdchem.BondType.SINGLE,
    Chem.rdchem.BondType.DOUBLE,
    Chem.rdchem.BondType.TRIPLE,
    Chem.rdchem.BondType.AROMATIC,
]

def one_hot(val, allowed):
    vec = [0] * (len(allowed) + 1)
    if val in allowed:
        vec[allowed.index(val)] = 1
    else:
        vec[-1] = 1
    return vec


def atom_features(atom):
    return (
        one_hot(atom.GetAtomicNum(), ATOM_FEATURES['atomic_num'])
        + one_hot(atom.GetDegree(), ATOM_FEATURES['degree'])
        + one_hot(atom.GetFormalCharge(), ATOM_FEATURES['formal_charge'])
        + one_hot(atom.GetTotalNumHs(), ATOM_FEATURES['num_hs'])
        + one_hot(atom.GetHybridization(), ATOM_FEATURES['hybridization'])
        + [1 if atom.GetIsAromatic() else 0]
    )


def bond_features(bond):
    return (
        one_hot(bond.GetBondType(), BOND_TYPES)
        + [1 if bond.GetIsConjugated() else 0]
        + [1 if bond.IsInRing() else 0]
    )


def smiles_to_graph(smiles, labels=None):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    x = [atom_features(atom) for atom in mol.GetAtoms()]
    x = torch.tensor(x, dtype=torch.float)

    edge_index = []
    edge_attr = []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bf = bond_features(bond)
        edge_index.extend([[i, j], [j, i]])
        edge_attr.extend([bf, bf])

    if len(edge_index) == 0:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr = torch.zeros((0, 7), dtype=torch.float)
    else:
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
        edge_attr = torch.tensor(edge_attr, dtype=torch.float)

    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)
    if labels is not None:
        data.y = torch.tensor(labels, dtype=torch.float)
    return data

sample = smiles_to_graph("CCO")
NODE_FEAT_DIM = sample.x.shape[1]
print(f"Node feature dim: {NODE_FEAT_DIM}")


Node feature dim: 145


In [8]:
# =====================
# 4) Fingerprints + descriptors embeddings
# =====================
# Descriptors: all RDKit descriptors
DESC_LIST = [d[0] for d in Descriptors._descList]

# Fingerprints
MORGAN_RADIUS = 2
MORGAN_NBITS = 2048

# NOTE: MACCS = substructure fingerprints (166 bits)

def smiles_to_fp_desc(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    # Descriptors
    desc_vals = []
    for name in DESC_LIST:
        fn = getattr(Descriptors, name)
        try:
            val = fn(mol)
        except Exception:
            val = np.nan
        desc_vals.append(val)
    desc_vals = np.array(desc_vals, dtype=np.float32)

    # MACCS keys (substructure FP)
    maccs = rdMolDescriptors.GetMACCSKeysFingerprint(mol)
    maccs = np.array(list(maccs.ToBitString()), dtype=np.int8)

    # Morgan (hashed) fingerprint
    morgan = rdMolDescriptors.GetMorganFingerprintAsBitVect(mol, radius=MORGAN_RADIUS, nBits=MORGAN_NBITS)
    morgan = np.array(list(morgan.ToBitString()), dtype=np.int8)

    # Concatenate: descriptors (float) + MACCS (0/1) + Morgan (0/1)
    return desc_vals, maccs, morgan


def build_fp_matrices(df, smiles_col):
    desc_list, maccs_list, morgan_list, keep_idx = [], [], [], []
    for i, smi in tqdm(list(enumerate(df[smiles_col].values)), total=len(df), desc="FP/Desc"):
        res = smiles_to_fp_desc(smi)
        if res is None:
            continue
        d, ma, mo = res
        desc_list.append(d)
        maccs_list.append(ma)
        morgan_list.append(mo)
        keep_idx.append(i)

    desc = np.vstack(desc_list)
    maccs = np.vstack(maccs_list)
    morgan = np.vstack(morgan_list)
    return desc, maccs, morgan, np.array(keep_idx)

# Build embeddings
train_desc, train_maccs, train_morgan, train_keep = build_fp_matrices(df_train, smiles_col)
test_desc, test_maccs, test_morgan, test_keep = build_fp_matrices(df_test, smiles_col)

# Align df with kept indices (skip molecules failing RDKit)
df_train = df_train.iloc[train_keep].reset_index(drop=True)
df_test = df_test.iloc[test_keep].reset_index(drop=True)

# Scale descriptors, leave bits as 0/1
scaler = StandardScaler()
train_desc_scaled = scaler.fit_transform(np.nan_to_num(train_desc, nan=0.0))
test_desc_scaled = scaler.transform(np.nan_to_num(test_desc, nan=0.0))

train_fp = np.hstack([train_desc_scaled, train_maccs, train_morgan]).astype(np.float32)
test_fp = np.hstack([test_desc_scaled, test_maccs, test_morgan]).astype(np.float32)

# L2 normalize for cosine similarity
train_fp_norm = normalize(train_fp, norm='l2')
test_fp_norm = normalize(test_fp, norm='l2')

FP_DIM = train_fp.shape[1]
print(f"Fingerprint/descriptor embedding dim: {FP_DIM}")


FP/Desc:   0%|          | 0/33668 [00:00<?, ?it/s][16:13:28] DEPRECATION WARNING: please use MorganGenerator
[16:13:28] DEPRECATION WARNING: please use MorganGenerator
[16:13:28] DEPRECATION WARNING: please use MorganGenerator
FP/Desc:   0%|          | 3/33668 [00:00<23:23, 23.98it/s][16:13:28] DEPRECATION WARNING: please use MorganGenerator
[16:13:28] DEPRECATION WARNING: please use MorganGenerator
[16:13:28] DEPRECATION WARNING: please use MorganGenerator
[16:13:28] DEPRECATION WARNING: please use MorganGenerator
[16:13:28] DEPRECATION WARNING: please use MorganGenerator
FP/Desc:   0%|          | 8/33668 [00:00<15:56, 35.21it/s][16:13:28] DEPRECATION WARNING: please use MorganGenerator
[16:13:28] DEPRECATION WARNING: please use MorganGenerator
[16:13:28] DEPRECATION WARNING: please use MorganGenerator
[16:13:28] DEPRECATION WARNING: please use MorganGenerator
[16:13:28] DEPRECATION WARNING: please use MorganGenerator
[16:13:28] DEPRECATION WARNING: please use MorganGenerator
FP/Desc:

Fingerprint/descriptor embedding dim: 2432


In [9]:
# =====================
# 5) Class prototypes for cosine similarity
# =====================
# Use mean embedding per class (on train set)
train_labels = df_train[label_cols].values.astype(np.int8)

class_means = np.zeros((num_classes, FP_DIM), dtype=np.float32)
for c in range(num_classes):
    idx = np.where(train_labels[:, c] == 1)[0]
    if len(idx) == 0:
        continue
    class_means[c] = train_fp_norm[idx].mean(axis=0)

# Normalize class means
class_means = normalize(class_means, norm='l2')

# Cosine similarity features (mean prototype)
# cosine = dot(normalized_fp, normalized_class_mean)
train_cos_mean = np.dot(train_fp_norm, class_means.T).astype(np.float32)
test_cos_mean = np.dot(test_fp_norm, class_means.T).astype(np.float32)

USE_NEAREST_COSINE = False  # set True if you want extra nearest-cosine features (expensive)
train_cos_nearest = None
test_cos_nearest = None

if USE_NEAREST_COSINE:
    from sklearn.neighbors import NearestNeighbors
    train_cos_nearest = np.zeros((len(train_fp_norm), num_classes), dtype=np.float32)
    test_cos_nearest = np.zeros((len(test_fp_norm), num_classes), dtype=np.float32)
    for c in range(num_classes):
        idx = np.where(train_labels[:, c] == 1)[0]
        if len(idx) < 2:
            continue
        # cosine distance = 1 - cosine similarity
        nbrs = NearestNeighbors(n_neighbors=1, metric='cosine')
        nbrs.fit(train_fp_norm[idx])
        dist_train, _ = nbrs.kneighbors(train_fp_norm)
        dist_test, _ = nbrs.kneighbors(test_fp_norm)
        train_cos_nearest[:, c] = 1.0 - dist_train[:, 0]
        test_cos_nearest[:, c] = 1.0 - dist_test[:, 0]

print("Cosine similarity features ready")


Cosine similarity features ready


In [17]:
# =====================
# 6) Build PyG graphs + attach FP/Cos features
# =====================

def build_graph_dataset(df, label_cols=None, fp=None, cos_mean=None, cos_nearest=None):
    graphs = []
    skipped = 0
    for i, row in tqdm(df.iterrows(), total=len(df), desc="Building graphs"):
        labels = row[label_cols].values.astype(np.float32) if label_cols else None
        g = smiles_to_graph(row[smiles_col], labels)
        if g is None:
            skipped += 1
            continue
        g.mol_id = row[id_col]
        if fp is not None:
            g.fp = torch.tensor(fp[i], dtype=torch.float)
        if cos_mean is not None:
            g.cos_mean = torch.tensor(cos_mean[i], dtype=torch.float)
        if cos_nearest is not None:
            g.cos_nearest = torch.tensor(cos_nearest[i], dtype=torch.float)
        graphs.append(g)
    print(f"Built {len(graphs)} graphs, skipped {skipped}")
    return graphs

train_graphs = build_graph_dataset(df_train, label_cols, train_fp, train_cos_mean, train_cos_nearest)
test_graphs = build_graph_dataset(df_test, label_cols=None, fp=test_fp, cos_mean=test_cos_mean, cos_nearest=test_cos_nearest)


Building graphs:   0%|          | 1/33668 [00:00<2:57:48,  3.16it/s][16:53:20] WARNING: not removing hydrogen atom without neighbors
[16:53:20] WARNING: not removing hydrogen atom without neighbors
[16:53:20] WARNING: not removing hydrogen atom without neighbors
Building graphs:  12%|█▏        | 4154/33668 [00:04<00:28, 1033.04it/s][16:53:24] WARNING: not removing hydrogen atom without neighbors
[16:53:24] WARNING: not removing hydrogen atom without neighbors
[16:53:24] WARNING: not removing hydrogen atom without neighbors
Building graphs:  14%|█▍        | 4839/33668 [00:04<00:26, 1095.10it/s][16:53:25] WARNING: not removing hydrogen atom without neighbors
[16:53:25] WARNING: not removing hydrogen atom without neighbors
[16:53:25] WARNING: not removing hydrogen atom without neighbors
Building graphs:  19%|█▉        | 6438/33668 [00:06<00:28, 949.71it/s] [16:53:27] WARNING: not removing hydrogen atom without neighbors
[16:53:27] WARNING: not removing hydrogen atom without neighbors
Buil

Built 33668 graphs, skipped 0


Building graphs:   8%|▊         | 852/11223 [00:00<00:07, 1420.56it/s][16:53:53] WARNING: not removing hydrogen atom without neighbors
[16:53:53] WARNING: not removing hydrogen atom without neighbors
[16:53:53] WARNING: not removing hydrogen atom without neighbors
[16:53:53] WARNING: not removing hydrogen atom without neighbors
[16:53:53] WARNING: not removing hydrogen atom without neighbors
Building graphs:  10%|█         | 1139/11223 [00:00<00:07, 1389.41it/s][16:53:53] WARNING: not removing hydrogen atom without neighbors
[16:53:53] WARNING: not removing hydrogen atom without neighbors
[16:53:53] WARNING: not removing hydrogen atom without neighbors
[16:53:53] Unusual charge on atom 6 number of radical electrons set to zero
Building graphs:  13%|█▎        | 1417/11223 [00:01<00:07, 1340.51it/s][16:53:53] WARNING: not removing hydrogen atom without neighbors
[16:53:53] WARNING: not removing hydrogen atom without neighbors
[16:53:53] WARNING: not removing hydrogen atom without neighbo

Built 11223 graphs, skipped 0


In [18]:
# =====================
# 7) Train/val split & loaders
# =====================
train_data, val_data = train_test_split(train_graphs, test_size=0.1, random_state=42)

BATCH_SIZE = 128
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_graphs, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_graphs)}")


Train: 30301, Val: 3367, Test: 11223


In [19]:
# =====================
# 8) Model: GIN + FP/Cos features
# =====================
class GINBlock(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        mlp = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.BatchNorm1d(out_dim),
            nn.ReLU(),
            nn.Linear(out_dim, out_dim),
        )
        self.conv = GINConv(mlp, train_eps=True)
        self.bn = nn.BatchNorm1d(out_dim)

    def forward(self, x, edge_index):
        x = self.conv(x, edge_index)
        x = self.bn(x)
        return F.relu(x)


class GINWithFP(nn.Module):
    def __init__(self, node_feat_dim, hidden_dim, num_classes, fp_dim, cos_dim, num_layers=5, dropout=0.3):
        super().__init__()
        self.num_layers = num_layers
        self.dropout = dropout
        self.input_proj = nn.Linear(node_feat_dim, hidden_dim)
        self.gin_layers = nn.ModuleList([GINBlock(hidden_dim, hidden_dim) for _ in range(num_layers)])
        self.jk_proj = nn.Linear(hidden_dim * (num_layers + 1), hidden_dim)

        self.fp_proj = nn.Linear(fp_dim, hidden_dim)
        self.cos_proj = nn.Linear(cos_dim, hidden_dim)

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 4, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes),
        )

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x = F.relu(self.input_proj(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        layer_outputs = [x]
        for gin in self.gin_layers:
            x = gin(x, edge_index)
            x = F.dropout(x, p=self.dropout, training=self.training)
            layer_outputs.append(x)
        x = torch.cat(layer_outputs, dim=-1)
        x = self.jk_proj(x)
        x_add = global_add_pool(x, batch)
        x_mean = global_mean_pool(x, batch)
        graph_rep = torch.cat([x_add, x_mean], dim=-1)

        # FP + cosine features
        fp_rep = F.relu(self.fp_proj(data.fp.view(data.num_graphs, -1)))
        cos_rep = F.relu(self.cos_proj(data.cos_mean.view(data.num_graphs, -1)))

        z = torch.cat([graph_rep, fp_rep, cos_rep], dim=-1)
        return self.classifier(z)


HIDDEN_DIM = 256
NUM_LAYERS = 5
DROPOUT = 0.3

model = GINWithFP(
    node_feat_dim=NODE_FEAT_DIM,
    hidden_dim=HIDDEN_DIM,
    num_classes=num_classes,
    fp_dim=FP_DIM,
    cos_dim=num_classes,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
).to(device)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")


Model parameters: 2,205,561


In [20]:
# =====================
# 9) Loss for imbalance
# =====================
all_labels = np.array([g.y.numpy() for g in train_graphs])
pos_freq = all_labels.mean(axis=0)
pos_freq = np.clip(pos_freq, 0.001, 0.999)
pos_weight = torch.tensor((1 - pos_freq) / pos_freq, dtype=torch.float).to(device)
pos_weight = torch.clamp(pos_weight, max=50.0)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
print(f"Pos weight range: [{pos_weight.min():.2f}, {pos_weight.max():.2f}]")


Pos weight range: [0.00, 50.00]


In [21]:
# =====================
# 10) Evaluation + hierarchy enforcement helpers
# =====================

def enforce_hierarchy_probs(probs, parent_mask_np):
    # enforce parent >= child by propagating max upwards
    # simple iterative pass
    out = probs.copy()
    changed = True
    while changed:
        changed = False
        for child_idx in range(parent_mask_np.shape[0]):
            parent_indices = np.where(parent_mask_np[child_idx] > 0)[0]
            for p in parent_indices:
                if out[:, p].min() < out[:, child_idx].min():
                    out[:, p] = np.maximum(out[:, p], out[:, child_idx])
                    changed = True
    return out


def select_max_class_and_prune(probs, parent_mask_np):
    # Choose the single max-prob class, set it to 1 and keep only its ancestors
    n = probs.shape[0]
    preds = np.zeros_like(probs, dtype=np.int8)

    # Precompute ancestor lists with DFS
    parents = {c: np.where(parent_mask_np[c] > 0)[0].tolist() for c in range(parent_mask_np.shape[0])}
    ancestors = {}
    for c in range(parent_mask_np.shape[0]):
        stack = parents.get(c, [])
        seen = set()
        while stack:
            p = stack.pop()
            if p in seen:
                continue
            seen.add(p)
            stack.extend(parents.get(p, []))
        ancestors[c] = list(seen)

    for i in range(n):
        c = int(np.argmax(probs[i]))
        preds[i, c] = 1
        for p in ancestors[c]:
            preds[i, p] = 1
    return preds


def evaluate(model, loader, threshold=0.5, enforce_hierarchy=False):
    model.eval()
    all_probs = []
    all_labels = []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            logits = model(batch)
            probs = torch.sigmoid(logits)
            all_probs.append(probs.cpu().numpy())
            labels = batch.y.view(probs.shape).cpu().numpy()
            all_labels.append(labels)

    all_probs = np.vstack(all_probs)
    all_labels = np.vstack(all_labels)

    if enforce_hierarchy:
        all_probs = enforce_hierarchy_probs(all_probs, parent_mask_np)

    all_preds = (all_probs >= threshold).astype(float)

    f1_micro = f1_score(all_labels, all_preds, average="micro", zero_division=0)
    f1_macro = f1_score(all_labels, all_preds, average="macro", zero_division=0)

    # violations
    violations = 0
    total_pairs = 0
    for child_idx in range(parent_mask_np.shape[0]):
        parent_indices = np.where(parent_mask_np[child_idx] > 0)[0]
        for p_idx in parent_indices:
            violations += ((all_preds[:, child_idx] == 1) & (all_preds[:, p_idx] == 0)).sum()
            total_pairs += len(all_preds)

    return f1_micro, f1_macro, int(violations), total_pairs, all_probs, all_preds, all_labels


In [22]:
# =====================
# 11) Training loop
# =====================
LR = 1e-3
EPOCHS = 50
PATIENCE = 8

optimizer = Adam(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)

best_f1 = 0
patience_counter = 0
history = {"train_loss": [], "val_f1_micro": [], "val_f1_macro": [], "violations": [], "viol_rate": []}

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        logits = model(batch)
        loss = criterion(logits, batch.y.view(logits.shape))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        total_loss += loss.item() * batch.num_graphs

    avg_loss = total_loss / len(train_data)

    f1_micro, f1_macro, violations, total_pairs, _, _, _ = evaluate(model, val_loader, enforce_hierarchy=False)
    scheduler.step(f1_micro)

    viol_rate = violations / total_pairs * 100 if total_pairs > 0 else 0
    history["train_loss"].append(avg_loss)
    history["val_f1_micro"].append(f1_micro)
    history["val_f1_macro"].append(f1_macro)
    history["violations"].append(violations)
    history["viol_rate"].append(viol_rate)

    print(f"Epoch {epoch:02d} | Loss: {avg_loss:.4f} | "
          f"F1-micro: {f1_micro:.4f} | F1-macro: {f1_macro:.4f} | "
          f"Violations: {violations:,} ({viol_rate:.2f}%)")

    if f1_micro > best_f1:
        best_f1 = f1_micro
        patience_counter = 0
        torch.save(model.state_dict(), "best_model_gin_fp.pt")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

print(f"Best validation F1-micro: {best_f1:.4f}")


RuntimeError: mat1 and mat2 shapes cannot be multiplied (1x311296 and 2432x256)

In [ ]:
# =====================
# 12) Curves
# =====================
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history["train_loss"])
axes[0].set_title("Train Loss")
axes[0].set_xlabel("Epoch")

axes[1].plot(history["val_f1_micro"], label="F1-micro")
axes[1].plot(history["val_f1_macro"], label="F1-macro")
axes[1].set_title("Validation F1")
axes[1].set_xlabel("Epoch")
axes[1].legend()

axes[2].plot(history["viol_rate"], color="red")
axes[2].set_title("Violation Rate % (val)")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("%")

plt.tight_layout()
plt.savefig("training_curves_gin_fp.png", dpi=150)
plt.show()


In [ ]:
# =====================
# 13) Inference + submission
# =====================
print("Loading best model for inference...")
model.load_state_dict(torch.load("best_model_gin_fp.pt", weights_only=True))
model.eval()

all_probs = []
all_mol_ids = []

with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        logits = model(batch)
        probs = torch.sigmoid(logits)
        all_probs.append(probs.cpu().numpy())
        all_mol_ids.extend(batch.mol_id)

all_probs = np.vstack(all_probs)

# Optional: enforce hierarchy on probabilities
all_probs = enforce_hierarchy_probs(all_probs, parent_mask_np)

# Option A: standard thresholding
preds = (all_probs >= 0.5).astype(int)

# Option B: single max-class + prune (set True to use)
USE_SINGLE_PATH = False
if USE_SINGLE_PATH:
    preds = select_max_class_and_prune(all_probs, parent_mask_np)

submission = pd.DataFrame(preds, columns=label_cols)
submission.insert(0, id_col, all_mol_ids)
submission = df_test[[id_col, smiles_col]].merge(submission, on=id_col, how="left")
submission[label_cols] = submission[label_cols].fillna(0).astype(int)

# Count violations
pred_vals = submission[label_cols].values
violations = 0
for child_idx in range(parent_mask_np.shape[0]):
    parent_indices = np.where(parent_mask_np[child_idx] > 0)[0]
    for p_idx in parent_indices:
        violations += ((pred_vals[:, child_idx] == 1) & (pred_vals[:, p_idx] == 0)).sum()

print(f"
Submission shape: {submission.shape}")
print(f"Hierarchy violations in test submission: {violations:,}")

submission.to_parquet("chebi_submission_sol1.parquet", index=False)
print("Saved to chebi_submission_sol1.parquet")
